In [ ]:
import os
from pathlib import Path

# Load HF_TOKEN from .env (lines look like `export HF_TOKEN=...`)
for line in Path('.env').read_text().splitlines():
    line = line.strip().removeprefix('export ')
    if '=' in line and not line.startswith('#'):
        key, value = line.split('=', 1)
        os.environ[key.strip()] = value.strip().strip('"\'')

from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    "CC-MAIN-2025-08",
    split="train",
    streaming=True,
)

In [ ]:
NUM_DOCS = 1000  # the stream is huge, so only take a sample

vespa_feed = []

for doc in dataset.take(NUM_DOCS):
    # FineWeb ids look like '<urn:uuid:d66bc6fe-...>'; strip the wrapper so it is a clean Vespa doc id
    doc_id = doc['id'].removeprefix('<urn:uuid:').removesuffix('>')
    vespa_feed.append({
        'id': doc_id,
        'fields': {
            'id': doc_id,
            'url': doc['url'],
            'text': doc['text'],
            'dump': doc['dump'],
            'year': int(doc['date'][:4]),  # date is e.g. '2013-05-18T05:48:54Z'
            'token_count': doc['token_count'],
        }
    })

print(len(vespa_feed))
print(vespa_feed[0]['fields']['url'])

In [ ]:
from vespa.application import ApplicationPackage

app_package = ApplicationPackage(name='fineweb')

In [ ]:
from vespa.package import Schema, Document

film_schema = Schema(name='fineweb_schema', document=Document())

In [ ]:
from vespa.package import Field

film_schema.add_fields(
    Field(
        name='url',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='text',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='dump',
        type='string',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='year',
        type='int',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='token_count',
        type='int',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='id',
        type='string',
        indexing=['summary']
    )
)

In [ ]:
from vespa.package import FieldSet, RankProfile

film_schema.add_field_set(
    FieldSet(
        name='default',
        fields=['url', 'text']
    ))

# Rank on both fields, boosting url matches a bit
film_schema.add_rank_profile(
    RankProfile(
        name='default',
        first_phase='bm25(text) + 2 * bm25(url)'
    ))

In [ ]:
app_package.add_schema(film_schema)

In [ ]:
from vespa.deployment import VespaCloud

vespa_cloud = VespaCloud(
    tenant='akashsinghal',
    application='fineweb',
    application_package=app_package
)

In [ ]:
app = vespa_cloud.deploy(region="aws-euw1-az1")

In [1]:
import os
from pathlib import Path
from vespa.application import Vespa

# Load .env so this cell works on its own after a restart
for line in Path('.env').read_text().splitlines():
    line = line.strip().removeprefix('export ')
    if '=' in line and not line.startswith('#'):
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"\''))

print(os.environ["VESPA_CERT_DIR"])

if "app" in globals():
    # Just deployed in this session
    cert_path = app.cert
    key_path = app.key
    endpoint = vespa_cloud.get_mtls_endpoint(region="aws-euw1-az1")
else:
    # Kernel restarted: reuse the already-deployed app
    cert_dir = Path(os.environ["VESPA_CERT_DIR"]).expanduser()
    cert_path = str(cert_dir / "data-plane-public-cert.pem")
    key_path = str(cert_dir / "data-plane-private-key.pem")
    endpoint = os.environ["VESPA_ENDPOINT"]

vespa_application = Vespa(endpoint, cert=cert_path, key=key_path)

~/.vespa/akashsinghal.fineweb.default


In [2]:
from vespa.io import VespaQueryResponse

def print_hits(response: VespaQueryResponse):
    for i, hit in enumerate(response.hits):
        f = hit['fields']
        print(f'{i+1:>3}  {f["id"][:8]}  {f["year"]}  {f["token_count"]:>6} tok  {f["url"]}')

In [5]:
# Searching for 'tornado' in the default fieldset (url + text)

response = vespa_application.query(
    yql="select * from sources * where text contains 'tornado'", # The default limit in Vespa is 10 results
)

print_hits(response)

In [6]:
# Using userQuery(): the query text is passed as a separate parameter

response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 100",
    query="tornado damage",
)

print_hits(response)

  1  c3aaf3f1  2025    2127 tok  https://furnishmycrib.co.uk/terms-conditions
  2  d7e18021  2025     538 tok  http://www.healthocity.us/the-10-best-resources-for-18/
  3  670c3955  2025     773 tok  https://blog-fruit-vegetable-ipm.extension.umn.edu/2023/07/japanese-beetle-fruit-crops-mostly-good.html
  4  99476645  2025    1290 tok  https://bycloudia.life/es/products/sterling-silver-cuff-bracelet-with-detailed-lattice-work
  5  4ba55192  2025     148 tok  https://gundamplacestore.com/products/re-100-gp04g-gerbera-water-decal
  6  55561115  2025    1781 tok  https://aingerroofing.ca/barrie-roofer-roofing-siding-attic-insulation/
  7  4aa11a5f  2025    2440 tok  https://carpetcleaningfortdodge.com/2020/11/how-to-prepare-your-home-for-winter-weather/
  8  67b290f2  2025     571 tok  http://rcplastering.co.uk/
  9  67777c70  2025     311 tok  https://fiberreinforcedpolymer.com/tag/frp-for-disaster-resilience-for-space-exploration-conference-2024/
 10  ce26ef07  2025     328 tok  https://

In [ ]:
# Search by text: matches words in the url or the body
response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 10",
    query="tornado damage",
)

print_hits(response)

In [ ]:
# Search by url: pass a full url or a fragment; punctuation is tokenized away
response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 10",
    query="http://ap.org/Content/Press-Release/2012/How-AP-reported-in-all-formats-from-tornado-stricken-regions",
    type="any",  # match any term, ranked by how many match
)

print_hits(response)

In [ ]:
# Restrict to one field when you only want url or only text
response = vespa_application.query(
    yql="select * from sources * where url contains 'ap' and text contains 'tornado' limit 10",
)

print_hits(response)